## 載入所需套件

匯入地理資料分析、遙測處理與視覺化相關的Python套件，建立分析環境。

In [1]:
# Phase 1: Environment Setup
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
import numpy as np
import os
from dotenv import load_dotenv
from urllib.parse import quote
import json
import rioxarray
from rasterstats import zonal_stats
# Load environment variables
load_dotenv('../.env')

# Check versions
print(f'geopandas {gpd.__version__} ✓')
print(f'pandas {pd.__version__} ✓')
print('All imports OK — ARIA system ready!')

Cannot find header.dxf (GDAL_DATA is not defined)


geopandas 1.1.3 ✓
pandas 2.3.3 ✓
All imports OK — ARIA system ready!


## 載入國土測繪中心鄉鎮市區界

從TGOS平台下載台灣各鄉鎮市區界線資料，設定座標系統為EPSG:3826。

In [2]:
# 1. 設定 TGOS 資料來源網址
TGOS_BASE = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/'
# 使用 quote 處理中文字元，避免網址解析錯誤
url = TGOS_BASE + quote('鄉(鎮、市、區)界線1140318.zip')
# 2. 讀取圖資 (指定 layer)
print("正在從 TGOS 載入圖資...")
townships = gpd.read_file(url, layer='TOWN_MOI_1140318')

# 3. 轉換座標系統至 EPSG:3826 (TWD97 121分帶)
townships = townships.to_crs(epsg=3826)

# 4. 印出檢查資訊
print(f"資料形狀 (Shape): {townships.shape}")
print(f"目前座標系統 (CRS): {townships.crs}")
display(townships.head(2))

c:\Users\Nitro\anaconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


正在從 TGOS 載入圖資...


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

## 篩選出花蓮縣

從全台鄉鎮資料中提取花蓮縣範圍，作為後續分析的目標區域。

In [ ]:
# 1. 篩選出花蓮縣的鄉鎮，並確保座標系統為 EPSG:3826 (公尺單位)
hualien_towns = townships[townships['COUNTYNAME'] == '花蓮縣'].to_crs(epsg=3826).copy()
hualien_towns.head(2)

## 載入河川資料

讀取水利署河川面圖資，檢查座標系統與幾何類型，準備進行空間分析。

In [ ]:
# Phase 2: Load River Data from Local File
# 載入水利署河川面圖資，檢查坐標系統與幾何類型。

print('Loading river polygons from local file...')
rivers = gpd.read_file('../data/riverpoly/riverpoly.shp')

print(f'CRS: {rivers.crs}')
print(f'Geometry type: {rivers.geom_type.unique()}')
rivers.head(2)

## 裁切目標河川

將全台河川資料以花蓮縣界線進行裁切，保留縣內河川片段以提升分析效率。

In [ ]:
# 1. 確保河川資料也是 EPSG:3826 (與花蓮鄉鎮資料一致)
if rivers.crs != "EPSG:3826":
    print("正在轉換河川資料座標系統...")
    rivers = rivers.to_crs(epsg=3826)

# 2. 將花蓮縣所有鄉鎮合併成一個完整的縣市範圍多邊形
# unary_union 會把所有鄉鎮的多邊形合併成一個大的幾何物件
hualien_boundary = hualien_towns.geometry.union_all()

# 3. 使用 clip 進行裁切
# 這會保留河川在花蓮縣境內的片段，並切掉超出縣界的部份
print("正在裁切花蓮縣河川資料...")
hualien_rivers = gpd.clip(rivers, hualien_boundary)

# 4. 檢查結果
print(f"原始河川數量: {len(rivers)}")
print(f"裁切後花蓮河川數量: {len(hualien_rivers)}")

# 視覺化檢查 (選用)
hualien_rivers.plot()

## 載入清理後的消防署避難收容所

讀取已清理的避難收容所資料，篩選座標有效記錄並轉換為GeoDataFrame格式。

In [ ]:
# Phase 3: Load and Use Cleaned Shelter Data
# 直接讀取已清理的避難收容所資料，只使用有效座標的記錄

print('Loading cleaned shelter data...')
shelters_csv_path = '../data/避難收容處所_清理後.csv'

if os.path.exists(shelters_csv_path):
    shelters_csv = pd.read_csv(shelters_csv_path, encoding='utf-8-sig')
    print(f'原始避難收容處所筆數： {len(shelters_csv)} ')
    
    # 只取座標有效性為"有效"的記錄
    if '座標有效性' in shelters_csv.columns:
        valid_shelters = shelters_csv[shelters_csv['座標有效性'] == '有效'].copy()
        print(f'清理後避難收容處所筆數： {len(valid_shelters)}')
        print(f'清理共 {len(shelters_csv) - len(valid_shelters)} 筆座標異常資料')
    else:
        print('Warning: "座標有效性" column not found, using all records')
        valid_shelters = shelters_csv.copy()
else:
    print(f'Warning: {shelters_csv_path} not found')

print(f'\n=== 清理後避難所數據 ===')
print(f'總有效避難收容處: {len(valid_shelters)}')
print(f'總有效收容人數: {valid_shelters["預計收容人數"].sum():,}')

# Convert to GeoDataFrame
shelters = gpd.GeoDataFrame(
    valid_shelters,
    geometry=gpd.points_from_xy(valid_shelters['經度'], valid_shelters['緯度']),
    crs='EPSG:4326'
)
# Convert to EPSG:3826 for analysis
shelters_3826 = shelters.to_crs(epsg=3826)
print(f'\n轉換後坐標系統:\n{shelters_3826.crs}')

shelters_3826.head(1)

## 取花蓮縣區域

從全台避難所資料中篩選花蓮縣範圍內的避難收容處所，準備進行後續分析。

In [ ]:
#取花蓮縣
hualien_shelters = shelters_3826[shelters_3826['縣市及鄉鎮市區'].str[:3] == '花蓮縣']
hualien_shelters.head(2)

## 載入tif檔案，內政部 20m DEM(花蓮縣)
載入dem_20m_hualien.tif，並且檢視檔案資訊

In [ ]:
import os
# 1. 設定檔案路徑 (請根據你的 Drive 資料夾結構微調)
# 定義兩個備選路徑
drive_path = '/content/drive/MyDrive/GIS_data/dem_20m_hualien.tif'
local_path = '../data/dem_20m_hualien.tif'

# 2. 判斷路徑並讀取檔案
if os.path.exists(drive_path):
    dem_path = drive_path
elif os.path.exists(local_path):
    dem_path = local_path

dem = rioxarray.open_rasterio(dem_path, masked=True)
# 將 CRS 標註為標準的 EPSG:3826
# inplace=True 代表直接修改原本的 dem 變數
# dem.rio.write_crs("EPSG:3826", inplace=True)

# 3. 提取並列印 Metadata
print("=== DEM Metadata 檢查結果 ===")
print(f"1. 數據形狀 (Bands, Y, X): {dem.shape}")
print(f"2. 座標系統 (CRS): {dem.rio.crs}")
print(f"3. 仿射轉換 (Transform):\n{dem.rio.transform()}")
print(f"4. 空間範圍 (Bounds): {dem.rio.bounds()}")

# 計算數值統計
# 使用 .values 轉換為 numpy 陣列計算，速度較快且處理 NaN
min_val = np.nanmin(dem.values)
max_val = np.nanmax(dem.values)
total_pixels = dem.size

print(f"5. 高程範圍 (Min/Max): {min_val:.2f} ~ {max_val:.2f} m")
print(f"6. 總像素量: {total_pixels:,} pixels")

## 轉換slope
透過公式計算花蓮縣的slope

In [ ]:
import numpy as np
dy, dx = np.gradient(dem.values[0], 20)  # 20m resolution
slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

## 讀取風險判定門檻閾值
確認風險門檻，從.env檔中讀取門檻值，確認坡度及海拔門檻

In [ ]:
# 讀環境變數
from dotenv import load_dotenv
import os
load_dotenv('../.env')
SLOPE_THRESHOLD = float(os.getenv('SLOPE_THRESHOLD'))
ELEVATION_LOW = float(os.getenv('ELEVATION_LOW'))
TARGET_COUNTY = os.getenv('TARGET_COUNTY')

print(f"系統啟動：目標區域 - {TARGET_COUNTY}")
print(f"風險門檻：坡度 > {SLOPE_THRESHOLD}°, 低海拔 < {ELEVATION_LOW}m")

## 讀取風險判定門檻-河川buffer資料
從.env檔中讀取門檻值，並對hualien_rivers進行.buffer，取兩個門檻(500m、1000m)，並儲存成GeoDataFrame

In [ ]:
buffer_high = float(os.getenv('BUFFER_HIGH'))
buffer_med = float(os.getenv('BUFFER_MED'))
# Create buffers (rivers already in EPSG:3826)
buffer_high_geom = hualien_rivers.buffer(buffer_high)
buffer_med_geom = hualien_rivers.buffer(buffer_med)

# Create GeoDataFrames for buffers
buffer_high_gdf = gpd.GeoDataFrame(
    {'risk_level': ['high'] * len(buffer_high_geom), 'buffer_m': [buffer_high] * len(buffer_high_geom)},
    geometry=buffer_high_geom,
    crs='EPSG:3826'
)

buffer_med_gdf = gpd.GeoDataFrame(
    {'risk_level': ['medium'] * len(buffer_med_geom), 'buffer_m': [buffer_med] * len(buffer_med_geom)},
    geometry=buffer_med_geom,
    crs='EPSG:3826'
)
# buffer_high_geom.to_crs(epsg=4326).plot()

## 統計避難所buffer後的區域統計資料
確認坐標系統後，對hualien_shelters(避難所)進行500m的.buffer，後續透過Zonal Statistic，取得mean_elevation、max_slope、std_elevation等資料

In [ ]:
# 1. 座標系統 (CRS) 檢查與對齊
# 確保避難所與 DEM 使用相同的投影系統 (EPSG:3826)
if hualien_shelters.crs != dem.rio.crs:
    print("正在對齊座標系統...")
    hualien_shelters = hualien_shelters.to_crs(dem.rio.crs)
print('shelter CRS match DEM CRS')

# 2. 建立 200m 緩衝區 (Buffer)
# 注意：在公尺單位(3826)下，buffer(200) 代表半徑 200 公尺
shelters_buffer = hualien_shelters.copy()
shelters_buffer['geometry'] = hualien_shelters.geometry.buffer(500)

# 3. 執行區域統計 (Zonal Statistics)
# 注意：雖然是用 shelters_buffer 計算，但結果會依序對應
dem_stats = zonal_stats(
    shelters_buffer, 
    dem.values[0], 
    affine=dem.rio.transform(),
    stats=['mean', 'std'], 
    nodata=np.nan
)

slope_stats = zonal_stats(
    shelters_buffer, 
    slope, # 使用之前計算好的 slope 陣列
    affine=dem.rio.transform(),
    stats=['max'], 
    nodata=np.nan
)

# 4. 回填資料至 hualien_shelters (點位版)
# 這樣你的 hualien_shelters 依然保留 Point 幾何，但多了地形統計欄位
hualien_shelters['mean_elevation'] = [s.get('mean') for s in dem_stats]
hualien_shelters['std_elevation'] = [s.get('std') for s in dem_stats]
hualien_shelters['max_slope'] = [s.get('max') for s in slope_stats]

# 5. 檢查結果
valid_count = hualien_shelters['mean_elevation'].notna().sum()
print(f"成功計算的避難所數量: {valid_count} / {len(hualien_shelters)}")

display(hualien_shelters.head(2))

## 為避難所建立風險指標
進行以下風險邏輯判定避難所風險層級
**風險分級邏輯**：
   - **極高風險**：距河川 < 500m **且** 最大坡度 > SLOPE_THRESHOLD
   - **高風險**：距河川 < 500m **或** 最大坡度 > SLOPE_THRESHOLD
   - **中風險**：距河川 < 1000m **且** 平均高程 < ELEVATION_LOW
   - **低風險**：其餘

並新增以下兩個欄位['risk_level', 'river_distance_category']

In [ ]:
# --- 1. 預先計算合併幾何 (優化效能) ---
# 避免在 apply 循環中重複執行耗時的 union_all()
union_500 = buffer_high_geom.geometry.union_all()
union_1000 = buffer_med_geom.geometry.union_all()

# --- 2. 修改分類函數 ---
def classify_risk_and_distance(row):
    # 取得基礎數值
    max_slope = row['max_slope']
    mean_elev = row['mean_elevation']
    geom = row['geometry']
    
    # 從環境變數或常數取得閥值
    SLOPE_THRESHOLD = float(os.getenv('SLOPE_THRESHOLD', 30))
    ELEVATION_LOW = float(os.getenv('ELEVATION_LOW', 100))
    
    # 空間判定
    is_within_500 = geom.within(union_500)
    is_within_1000 = geom.within(union_1000)
    
    # --- 判定河川距離分類 (river_distance_category) ---
    if is_within_500:
        dist_cat = '<500m'
    elif is_within_1000:
        dist_cat = '<1000m'
    else:
        dist_cat = '>1000m'
    
    # --- 判定風險邏輯 (risk_level) ---
    if is_within_500 and max_slope > SLOPE_THRESHOLD:
        risk = '極高風險'
    elif is_within_500 or max_slope > SLOPE_THRESHOLD:
        risk = '高風險'
    elif is_within_1000 and mean_elev < ELEVATION_LOW:
        risk = '中風險'
    else:
        risk = '低風險'
    
    # 回傳 Pandas Series 以前往多個欄位
    return pd.Series([risk, dist_cat])

# --- 3. 執行分類並新增欄位 ---
# 使用 apply 一次產出兩個新欄位
hualien_shelters[['risk_level', 'river_distance_category']] = hualien_shelters.apply(classify_risk_and_distance, axis=1)

# 檢視前兩筆結果
hualien_shelters.head(2)

# 將避難所相關資訊儲存成.json，並察看結果

In [ ]:
# 1. 轉成一般的 DataFrame 並移除 geometry 欄位
df_to_save = pd.DataFrame(hualien_shelters.drop(columns='geometry'))

# 2. 現在可以使用 pandas 的 to_json，它支援路徑參數
df_to_save.to_json("../outputs/terrain_risk_audit.json", 
                   orient='records', 
                   force_ascii=False, 
                   indent=4)

print("檔案已成功儲存為標準 JSON 表格：terrain_risk_audit.json")

# 讀取 JSON 檔案
df_audit = pd.read_json("../outputs/terrain_risk_audit.json")

# 查看前幾筆資料
print("資料讀取成功！")
display(df_audit.head(2))

# 繪製DEM + 避難所風險地圖
將中文字體設定為微軟正黑體，並進行以下方式繪圖
### 1. 座標系統同步 (CRS Alignment)
為了確保製圖的精準度與未來的數據擴充性（如疊加 GPS 即時位移），本步驟首先將行政區（`hualien_towns`）與避難處所（`hualien_shelters`）統一投影至 **WGS84 (EPSG:4326)**。這確保了向量層與光柵化的 DEM 數據在經緯度框架下完美重合，消除了因投影落差產生的分析誤差。

### 2. 立體地形底圖：陰影與高程的藝術
這是我在視覺化過程中克服的核心技術挑戰。為了增加地圖的直觀感受，我們不只使用單純的高程圖，還引入了 **Hillshade（山體陰影）**：
* **技術突破 (Masking Strategy)**：針對 DEM 邊界外的「海域」或「無資料區」，我們透過 `np.isnan()` 建立遮罩。在計算陰影時雖補為 0 以利數學運算，但最終顯色前將其還原為 **NaN (透明)**，徹底解決了先前背景變藍色的問題。
* **視覺優化**：疊加 `terrain` 色票並配合 `alpha=0.6`。這能讓中央山脈的深邃河谷與縱谷平原的平緩特徵同時透過陰影展現立體感，使指揮官能一眼辨認出「高海拔但平緩」與「低海拔但陡峭」的區域差異。

### 3. 多層次繪圖策略 (Layering & Z-Order)
我們採用了嚴格的層次管理，確保關鍵資訊不被地形底圖掩沒：
* **底層 (`z=0`)**：黑白 Hillshade，提供地形骨架與光影質感。
* **中底層 (`z=0.5`)**：彩色 DEM，賦予海拔高度意義。
* **中層 (`z=1`)**：行政區邊界，提供基礎地理脈絡與行政界線。
* **頂層 (`z=10`)**：避難所點位，根據 ARIA 邏輯產出的四個風險等級（極高、高、中、低）進行分色。

### 4. 決策支援與輸出
最終圖表以 **300 DPI** 高解析度儲存至 `../outputs/terrain_risk_map.png`。
* **分析小結**：透過立體圖可以觀察到，多數「極高風險（紅色）」站點分佈於縱谷邊緣與山脈的交接處（沖積扇頂部或山腳陡坡），這驗證了地形因子在避難安全性評估中的主導地位。

---

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from rasterio.enums import Resampling
from matplotlib.colors import LightSource

# --- 1. 字體與環境設定 ---
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] 
plt.rcParams['axes.unicode_minus'] = False

# --- 2. 圖層轉換 (WGS84) ---
# 確保向量資料與 DEM 座標一致
t_4326 = hualien_towns.to_crs(epsg=4326)
s_4326 = hualien_shelters.to_crs(epsg=4326)

# --- 3. 準備畫布 ---
fig, ax = plt.subplots(figsize=(12, 16))
ax.set_facecolor('white') # 確保背景是乾淨的白色

# --- 4. 疊合 DEM + Hillshade (修正藍色背景關鍵) ---
try:
    # A. 座標轉換
    dem_4326 = dem.rio.reproject("EPSG:4326", resampling=Resampling.bilinear)
    
    # B. 取得數值並建立遮罩 (保留 NaN)
    elevation = dem_4326.values[0] if dem_4326.ndim == 3 else dem_4326.values
    elevation = elevation.astype(float) # 確保為浮點數以支援 NaN
    
    # 建立 NaN 遮罩，記錄哪裡是「沒資料」的地方
    mask = np.isnan(elevation)
    
    # C. 計算 Hillshade 
    # 計算陰影時暫時填補 0 (因為 LightSource 不支援 NaN 計算)
    ls = LightSource(azdeg=315, altdeg=45)
    hillshade = ls.hillshade(np.nan_to_num(elevation, nan=0), vert_exag=1)
    
    # 重要：將陰影中原本是 NaN 的位置變回 NaN，這樣背景才會透明
    hillshade[mask] = np.nan
    
    # D. 取得空間範圍
    ext = [dem_4326.x.min(), dem_4326.x.max(), dem_4326.y.min(), dem_4326.y.max()]
    
    # E. 繪製圖層
    # 先畫 Hillshade (黑白底圖)
    ax.imshow(hillshade, extent=ext, cmap='gray', origin='upper', zorder=0)
    
    # 再畫彩色 DEM (因為 elevation 含有 NaN，背景不會變藍色)
    img = ax.imshow(
        elevation,
        cmap='terrain',
        alpha=0.6, # 透明度讓陰影透出來
        extent=ext,
        origin='upper',
        zorder=0.5
    )
    
    # 加入 Colorbar
    cbar = fig.colorbar(img, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label('Elevation (m)')

except Exception as e:
    print(f"DEM/Hillshade 繪製錯誤: {e}")

# --- 5. 繪製向量圖層 ---
# 行政區邊界
t_4326.plot(ax=ax, color='none', edgecolor='black', linewidth=0.8, alpha=0.6, zorder=1)

# 避難所 (依風險等級著色)
risk_colors = {'極高風險': 'red', '高風險': 'orange', '中風險': 'yellow', '低風險': 'blue'}
for level, color in risk_colors.items():
    subset = s_4326[s_4326['risk_level'] == level]
    if not subset.empty:
        subset.plot(ax=ax, color=color, markersize=50, edgecolor='black', 
                    linewidth=0.7, label=level, zorder=10)

# --- 6. 座標軸設定 ---
b = t_4326.total_bounds
ax.set_xlim(b[0], b[2])
ax.set_ylim(b[1], b[3])
ax.ticklabel_format(useOffset=False, style='plain')
plt.xticks(rotation=45)

# --- 7. 圖例與標題 ---
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markeredgecolor='black', markersize=10, label='極高風險'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markeredgecolor='black', markersize=10, label='高風險'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow', markeredgecolor='black', markersize=10, label='中風險'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markeredgecolor='black', markersize=10, label='低風險')
]

plt.title("DEM + 避難所地圖(花蓮縣)", fontsize=16)
ax.legend(handles=legend_elements, loc='upper left', shadow=True, title="避難所風險等級")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
# 儲存
save_path = '../outputs/terrain_risk_map.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"圖片已儲存至: {save_path}")

plt.show()

## 繪製Top 10 高風險避難所的坡度 vs. 高程散佈圖
取出前 10 名高風險避難所 (假設以最大坡度排序)，並繪製Top 10 高風險避難所的坡度 vs. 高程散佈圖，最後將圖片儲存

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. 設定中文字體 (微軟正黑體)
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

# 2. 篩選資料：取得前 10 名高風險避難所 (假設以最大坡度排序)
# 如果你有極高風險，建議先篩選極高風險
top_10 = hualien_shelters[hualien_shelters['risk_level'].isin(['極高風險', '高風險'])] \
            .sort_values(by='max_slope', ascending=False) \
            .head(10)

# 3. 建立畫布
plt.figure(figsize=(10, 7))

# 4. 繪製散佈圖
# X軸：平均高程, Y軸：最大坡度
plt.scatter(top_10['mean_elevation'], top_10['max_slope'], 
            c='red', s=100, alpha=0.7, edgecolors='black', label='高風險避難所')

# 5. 為每個點加上名稱標籤 (文字標註)
for i, row in top_10.iterrows():
    plt.annotate(row['避難收容處所名稱'], 
                 (row['mean_elevation'], row['max_slope']),
                 xytext=(8, 5),             # 文字偏移位置
                 textcoords='offset points', 
                 fontsize=10,
                 alpha=0.8)

# 6. 設定圖表裝飾
plt.title('Top 10 高風險避難所：坡度 vs. 高程分析', fontsize=16, pad=15)
plt.xlabel('平均高程 (m)', fontsize=12)
plt.ylabel('最大坡度 (°)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

# 加入一條坡度 30 度的警戒線
plt.axhline(y=30, color='orange', linestyle=':', label='風險閾值 (30°)')

plt.legend()

# 7. 儲存圖片
save_path = '../outputs/top10_risk_scatter.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"統計圖已儲存至: {save_path}")

plt.show()